In [3]:
%cd /content/drive/MyDrive/ASR

/content/drive/MyDrive/ASR


In [4]:
import argparse
import os
import sys

import torch
import librosa
import numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from pathlib import Path

# ── Defaults ──────────────────────────────────────────────────────────────────
DEFAULT_MODEL     = "./whisper-small-librispeech"   # fine-tuned checkpoint
FALLBACK_MODEL    = "openai/whisper-small"           # if checkpoint not found
LANGUAGE          = "english"
TASK              = "transcribe"
SAMPLING_RATE     = 16_000
MAX_NEW_TOKENS    = 225
CHUNK_SECONDS     = 30      # Whisper's context window
TIME_SLOT_WINDOW  = 5.0     # Group output by 5-second slots
# ──────────────────────────────────────────────────────────────────────────────


def load_model(model_path: str):
    """Load processor + model from a local path or HF Hub id."""
    print(f"Loading model from '{model_path}' …")
    processor = WhisperProcessor.from_pretrained(model_path, language=LANGUAGE, task=TASK)
    model     = WhisperForConditionalGeneration.from_pretrained(model_path)
    model.eval()
    if torch.cuda.is_available():
        model = model.to("cuda")
        print("  Running on GPU ✓")
    else:
        print("  Running on CPU (slower)")
    return processor, model


def load_audio(path: str) -> np.ndarray:
    """Load any audio file and resample to 16 kHz mono."""
    if not os.path.isfile(path):
        sys.exit(f"[ERROR] File not found: {path}")
    print(f"Loading audio: {path}")
    waveform, _ = librosa.load(path, sr=SAMPLING_RATE, mono=True)
    duration = len(waveform) / SAMPLING_RATE
    print(f"  Duration : {duration:.2f} s  |  Samples : {len(waveform)}")
    return waveform


def parse_rttm(rttm_path: str):
    """Parse RTTM diarization file and return list of speaker segments.

    RTTM format: SPEAKER file 1 start_time duration <NA> <NA> speaker <NA> <NA>

    Returns:
        List of dicts: [{"start": float, "end": float, "duration": float, "speaker": str}, ...]
    """
    segments = []
    if not os.path.isfile(rttm_path):
        print(f"[WARN] RTTM file not found: {rttm_path}")
        return segments

    print(f"Parsing diarization from: {rttm_path}")
    with open(rttm_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 8 and parts[0] == "SPEAKER":
                start_time = float(parts[3])
                duration = float(parts[4])
                speaker = parts[7]
                segments.append({
                    "start": start_time,
                    "end": start_time + duration,
                    "duration": duration,
                    "speaker": speaker
                })

    print(f"  Found {len(segments)} speaker segments")
    return segments


def transcribe_chunk(processor, model, audio_chunk: np.ndarray) -> str:
    """Transcribe a single ≤30-second chunk."""
    device = next(model.parameters()).device
    inputs = processor(audio_chunk, sampling_rate=SAMPLING_RATE, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()


def transcribe(processor, model, waveform: np.ndarray) -> str:
    """
    Handle audio of any length by splitting into CHUNK_SECONDS chunks
    and concatenating the results.
    """
    chunk_len   = CHUNK_SECONDS * SAMPLING_RATE
    n_chunks    = int(np.ceil(len(waveform) / chunk_len))
    transcripts = []

    if n_chunks == 1:
        return transcribe_chunk(processor, model, waveform)

    print(f"  Audio > {CHUNK_SECONDS}s — splitting into {n_chunks} chunks …")
    for i in range(n_chunks):
        chunk = waveform[i * chunk_len : (i + 1) * chunk_len]
        print(f"  Transcribing chunk {i+1}/{n_chunks} …", end=" ", flush=True)
        text = transcribe_chunk(processor, model, chunk)
        print(f"→ '{text[:60]}{'…' if len(text)>60 else ''}'")
        transcripts.append(text)

    return " ".join(transcripts)


def extract_audio_segment(waveform, start_time, end_time, sr=SAMPLING_RATE):
    """Extract audio segment between start_time and end_time."""
    start_sample = int(start_time * sr)
    end_sample = int(end_time * sr)
    return waveform[start_sample:end_sample]


def transcribe_speaker_segments(processor, model, waveform, segments, sr=SAMPLING_RATE):
    """Transcribe each speaker segment and return list with transcripts.

    Returns:
        List of dicts: [{"start": float, "end": float, "speaker": str, "text": str}, ...]
    """
    results = []

    print(f"\nTranscribing {len(segments)} speaker segments …")
    for i, seg in enumerate(segments):
        start = seg["start"]
        end = seg["end"]
        speaker = seg["speaker"]

        # Extract audio for this segment
        audio_segment = extract_audio_segment(waveform, start, end, sr)

        if len(audio_segment) == 0:
            text = "[silence]"
        else:
            # Transcribe this segment
            print(f"  [{i+1}/{len(segments)}] {speaker} ({start:.2f}s-{end:.2f}s) …", end=" ", flush=True)
            text = transcribe(processor, model, audio_segment)
            print(f"✓ '{text[:50]}{'…' if len(text)>50 else ''}'")

        results.append({
            "start": start,
            "end": end,
            "speaker": speaker,
            "text": text
        })

    return results


def format_speaker_transcripts_by_slots(transcripts, time_window=TIME_SLOT_WINDOW):
    """Format speaker transcripts grouped by time slots.

    Returns:
        Formatted string organized by time slots with speaker transcripts
    """
    if not transcripts:
        return "No transcripts available"

    # Find max time
    max_time = max(t["end"] for t in transcripts)
    num_slots = int(np.ceil(max_time / time_window))

    output = []
    output.append(f"\nSPEAKER DIARIZATION WITH TRANSCRIPTS (window: {time_window}s)\n")
    output.append("=" * 70)

    for slot_idx in range(num_slots):
        slot_start = slot_idx * time_window
        slot_end = (slot_idx + 1) * time_window

        output.append(f"\nTime Slot {slot_idx + 1}: {slot_start:.1f}s - {slot_end:.1f}s")
        output.append("-" * 70)

        # Find transcripts in this time slot
        speakers_in_slot = {}
        for t in transcripts:
            # Check if segment overlaps with this time slot
            if t["start"] < slot_end and t["end"] > slot_start:
                speaker = t["speaker"]
                if speaker not in speakers_in_slot:
                    speakers_in_slot[speaker] = []
                speakers_in_slot[speaker].append(t)

        if not speakers_in_slot:
            output.append("(No speech)")
        else:
            # Sort speakers by name
            for speaker in sorted(speakers_in_slot.keys()):
                items = speakers_in_slot[speaker]
                output.append(f"\n{speaker}:")
                for item in items:
                    output.append(f"  [{item['start']:.2f}s - {item['end']:.2f}s]")
                    output.append(f"  \"{item['text']}\"")

    return "\n".join(output)


def main(args_list=None):
    parser = argparse.ArgumentParser(description="Whisper ASR with Speaker Diarization")
    parser.add_argument(
        "--audio", required=True,
        help="Path to a WAV (or any audio) file to transcribe"
    )
    parser.add_argument(
        "--rttm", required=True,
        help="Path to RTTM diarization file with speaker segments"
    )
    parser.add_argument(
        "--model", default=None,
        help=(
            f"Model path or HF Hub id. "
            f"Defaults to '{DEFAULT_MODEL}' if it exists, "
            f"otherwise falls back to '{FALLBACK_MODEL}'."
        )
    )
    args = parser.parse_args(args_list)

    # Resolve model path
    if args.model:
        model_path = args.model
    elif os.path.isdir(DEFAULT_MODEL):
        model_path = DEFAULT_MODEL
    else:
        print(f"[WARN] '{DEFAULT_MODEL}' not found — using base model '{FALLBACK_MODEL}'")
        model_path = FALLBACK_MODEL

    # Load model & audio
    processor, model = load_model(model_path)
    waveform = load_audio(args.audio)

    # Parse diarization
    segments = parse_rttm(args.rttm)
    if not segments:
        sys.exit("[ERROR] No segments found in RTTM file")

    # Transcribe speaker segments
    print("Transcribing …")
    transcripts = transcribe_speaker_segments(processor, model, waveform, segments)

    # Format output
    formatted_output = format_speaker_transcripts_by_slots(transcripts, time_window=TIME_SLOT_WINDOW)

    # Display output
    print("\n" + "=" * 70)
    print(formatted_output)
    print("=" * 70)

    # Save to .txt
    out_txt = os.path.splitext(args.audio)[0] + "_speaker_transcript.txt"
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(formatted_output)
    print(f"\nSaved speaker transcript → {out_txt}")


if __name__ == "__main__":
    if 'ipykernel' in sys.modules:
        pass
    else:
        main()


In [6]:
main(args_list=["--audio", "/content/drive/MyDrive/ASR/sample_0.wav", "--rttm", "/content/drive/MyDrive/ASR/diarization_output.rttm"])

Loading model from './whisper-small-librispeech' …


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  Running on GPU ✓
Loading audio: /content/drive/MyDrive/ASR/sample_0.wav
  Duration : 178.78 s  |  Samples : 2860513
Parsing diarization from: /content/drive/MyDrive/ASR/diarization_output.rttm
  Found 46 speaker segments
Transcribing …

Transcribing 46 speaker segments …
  [1/46] SPEAKER_03 (0.01s-33.00s) …   Audio > 30s — splitting into 2 chunks …
  Transcribing chunk 1/2 … 

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformer

→ 'DEBATE IS HEATING UP OVER WHETHER OR NOT THE FEARLESS GIRL S…'
  Transcribing chunk 2/2 … 

...

✓ '[BLANK_AUDIO]'
  [45/46] SPEAKER_04 (177.61s-177.66s) … 

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ 'THE'
  [46/46] SPEAKER_02 (178.31s-178.79s) … 

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ 'Well today'


SPEAKER DIARIZATION WITH TRANSCRIPTS (window: 5.0s)


Time Slot 1: 0.0s - 5.0s
----------------------------------------------------------------------

SPEAKER_03:
  [0.01s - 33.00s]
  "DEBATE IS HEATING UP OVER WHETHER OR NOT THE FEARLESS GIRL STATUE SHOULD STAY ON WALL STREET THE STATUE WAS PUT UP TO HIGHLIGHT INTERNATIONAL WOMEN'S DAY AND IT IMMEDIATELY BECAME A TOURIST DRAW AND AN INTERNET SENSATION THE FEARLESS GIRL IS RIGHT ACROSS THE ICONIC WALL STREET CHARGING BULL SO ORIGINALLY THIS WAS PUT UP ONLY UNTIL APRIL SECOND BUT JUST YESTERDAY THE MAYOR'S OFFICE SAID WE'RE ACTUALLY GOING TO LET THIS STAY UNTIL FEB. OF 2018 SO THIS WAS KIND OF MEANT IT WENT IN THIS LOCATION SPECIFICALLY BECAUSE THEY WANT TO ENCOURAGE MORE PEOPLE OR MORE FIRMS ON WALL STREET TO HIGHLIGHT HIGHER MORE WOMEN SO THAT'S KIND OF WHERE THIS GUTS START"

Time Slot 2: 5.0s - 10.0s
----------------------------------------------------------------------

SPEAKER_03:
  [0.01s - 33.00s]
  "DEBATE IS H

In [ ]:
import argparse
import os
import sys

import torch
import librosa
import numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# ── Defaults ──────────────────────────────────────────────────────────────────
DEFAULT_MODEL     = "./whisper-small-librispeech"   # fine-tuned checkpoint
FALLBACK_MODEL    = "openai/whisper-small"           # if checkpoint not found
LANGUAGE          = "english"
TASK              = "transcribe"
SAMPLING_RATE     = 16_000
MAX_NEW_TOKENS    = 225
CHUNK_SECONDS     = 30      # Whisper's context window
# ──────────────────────────────────────────────────────────────────────────────


def load_model(model_path: str):
    """Load processor + model from a local path or HF Hub id."""
    print(f"Loading model from '{model_path}' …")
    processor = WhisperProcessor.from_pretrained(model_path, language=LANGUAGE, task=TASK)
    model     = WhisperForConditionalGeneration.from_pretrained(model_path)
    model.eval()
    if torch.cuda.is_available():
        model = model.to("cuda")
        print("  Running on GPU ✓")
    else:
        print("  Running on CPU (slower)")
    return processor, model


def load_audio(path: str) -> np.ndarray:
    """Load any audio file and resample to 16 kHz mono."""
    if not os.path.isfile(path):
        sys.exit(f"[ERROR] File not found: {path}")
    print(f"Loading audio: {path}")
    waveform, _ = librosa.load(path, sr=SAMPLING_RATE, mono=True)
    duration = len(waveform) / SAMPLING_RATE
    print(f"  Duration : {duration:.2f} s  |  Samples : {len(waveform)}")
    return waveform


def transcribe_chunk(processor, model, audio_chunk: np.ndarray) -> str:
    """Transcribe a single ≤30-second chunk."""
    device = next(model.parameters()).device
    inputs = processor(audio_chunk, sampling_rate=SAMPLING_RATE, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()


def transcribe(processor, model, waveform: np.ndarray) -> str:
    """
    Handle audio of any length by splitting into CHUNK_SECONDS chunks
    and concatenating the results.
    """
    chunk_len   = CHUNK_SECONDS * SAMPLING_RATE
    n_chunks    = int(np.ceil(len(waveform) / chunk_len))
    transcripts = []

    if n_chunks == 1:
        return transcribe_chunk(processor, model, waveform)

    print(f"  Audio > {CHUNK_SECONDS}s — splitting into {n_chunks} chunks …")
    for i in range(n_chunks):
        chunk = waveform[i * chunk_len : (i + 1) * chunk_len]
        print(f"  Transcribing chunk {i+1}/{n_chunks} …", end=" ", flush=True)
        text = transcribe_chunk(processor, model, chunk)
        print(f"→ '{text[:60]}{'…' if len(text)>60 else ''}'")
        transcripts.append(text)

    return " ".join(transcripts)


def main(args_list=None): # Modified: added args_list parameter
    parser = argparse.ArgumentParser(description="Whisper Small ASR inference")
    parser.add_argument(
        "--audio", required=True,
        help="Path to a WAV (or any audio) file to transcribe"
    )
    parser.add_argument(
        "--model", default=None,
        help=(
            f"Model path or HF Hub id. "
            f"Defaults to '{DEFAULT_MODEL}' if it exists, "
            f"otherwise falls back to '{FALLBACK_MODEL}'."
        )
    )
    args = parser.parse_args(args_list) # Modified: passed args_list to parse_args

    # Resolve model path
    if args.model:
        model_path = args.model
    elif os.path.isdir(DEFAULT_MODEL):
        model_path = DEFAULT_MODEL
    else:
        print(f"[WARN] '{DEFAULT_MODEL}' not found — using base model '{FALLBACK_MODEL}'")
        model_path = FALLBACK_MODEL

    # Load model & audio
    processor, model = load_model(model_path)
    waveform         = load_audio(args.audio)

    # Transcribe
    print("Transcribing …")
    transcript = transcribe(processor, model, waveform)

    # Output
    print("\n" + "─" * 60)
    print("TRANSCRIPT:")
    print(transcript)
    print("─" * 60)

    # Optionally save to .txt
    out_txt = os.path.splitext(args.audio)[0] + "_transcript.txt"
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(transcript)
    print(f"\nSaved transcript → {out_txt}")


if __name__ == "__main__":
    # This allows calling main() with arguments from the notebook
    # while still allowing command-line execution when the script is run directly.
    if 'ipykernel' in sys.modules:
        # In an IPython environment (like Colab), do nothing here.
        # The user will call main() with specific args in a subsequent cell.
        pass
    else:
        # When run as a script from the command line, use sys.argv
        main()

In [ ]:
main(args_list=["--audio", "/content/drive/MyDrive/ASR/bxcfq.wav"])

Loading model from './whisper-small-librispeech' …


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  Running on CPU (slower)
Loading audio: /content/drive/MyDrive/ASR/bxcfq.wav
  Duration : 196.48 s  |  Samples : 3143680
Transcribing …
  Audio > 30s — splitting into 7 chunks …
  Transcribing chunk 1/7 … 

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformer

→ 'I DO WANT TO TALK ABOUT IMAGRATION AND ILLIGAL IMAGRATION IN…'
  Transcribing chunk 2/7 … 

...

→ 'BUT WE'LL LOOK VERY VERY STRONGLY AT WHAT WE DID'
  Transcribing chunk 6/7 … 

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


→ 'WELL'
  Transcribing chunk 7/7 … 

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


→ 'subject had a very complicated subject.IWANT IT I WANT IT TH…'

────────────────────────────────────────────────────────────
TRANSCRIPT:
I DO WANT TO TALK ABOUT IMAGRATION AND ILLIGAL IMAGRATION IN ITUAL THAT YOU'VE TALKED ABOUT A LOT YOU'RE AT THE BORDER THE OTHER DAY ASSUMING A PRESIDENT TRUMP IS ABLE TO STOP THE FLOW OF ILLIGAL IMAGRATION THROUGH BUILDING A WALL OR SOME PARTS OF A WALL WHAT DO YOU THINK SHOULD BE DONE WITH THE ESTIMATED 11 MILLION UNDOCUMENTED WORKERS AND THEIR FAMILY IS ALREADY HERE WOULD YOU BE OPEN MINDED ABOUT A PATH TO CITIZENShip IS AT A NONSTARTABLE WITH YOU WHERE ARE YOU ON THAT ISSU IT WILL ACTUALLY SAVE MONEY BY DOING IT AND DOING IT PROPERLY ONCE THAT'S DONE WE HAVE A SITUATION THAT IS GOING TO BE DONE IMMEDIATELY WHAT BEFORE THAT'S DONE WE'RE GONNA GET THE BAD ONES OUT WE HAVE SOME REALLY BAD DUDES RIGHT HERE IN THIS COUNTRY AND WE'RE GETTING THEM OUT AND WE'RE SENDING HER BACK TO WHERE THEY CAME FROM AND I DON'T MEAN MEXICOLE I MEAN IT'S THEY COME FR